In [35]:
import pandas as pd
import torch

In [36]:
#read csv: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
veri = pd.read_csv("IMDB Dataset.csv")


In [37]:
veri.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [38]:
print(veri.columns)
print(veri.shape)
print(veri["sentiment"].value_counts())
veri_kucuk = veri.sample(n=2000, random_state=42).reset_index(drop=True)

Index(['review', 'sentiment'], dtype='str')
(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [39]:
veri_kucuk["label"] = veri_kucuk["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [40]:
veri_kucuk[["review", "sentiment", "label"]].head()

,review,sentiment,label
0,I really liked this Summerslam due to the look...,positive,1
1,Not many television shows appeal to quite as m...,positive,1
2,The film quickly gets to a major chase scene w...,negative,0
3,Jane Austen would definitely approve of this o...,positive,1
4,Expectations were somewhat high for me when I ...,negative,0


In [41]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    veri_kucuk["review"].tolist(),
    veri_kucuk["label"].tolist(),
    test_size=0.2,
    random_state=44,
    stratify=veri_kucuk["label"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=44,
    stratify=temp_labels
)

In [42]:
#from transformers import AutoTokenizer, AutoModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained( model_name, num_labels=2)


#tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
#model = AutoModel.from_pretrained("bert-base-uncased")

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [43]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [44]:
train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

In [45]:
import torch

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [46]:
train_dataset = IMDBDataset(train_encodings, train_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

In [47]:
from transformers import TrainingArguments, Trainer

In [48]:
training_args = TrainingArguments(
    output_dir="./bert_imdb_output",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [49]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [50]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [51]:
trainer.train()

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.325989,0.369517,0.850000,0.769841,0.989796,0.866071
2,0.266962,0.272005,0.915000,0.893204,0.938776,0.915423
3,0.071310,0.391279,0.905000,0.915789,0.887755,0.901554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=600, training_loss=0.31641422073046366, metrics={'train_runtime': 5671.4045, 'train_samples_per_second': 0.846, 'train_steps_per_second': 0.106, 'total_flos': 631466532864000.0, 'train_loss': 0.31641422073046366, 'epoch': 3.0})

In [52]:
trainer.evaluate()


c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.071310,0.272487,3,0.915000,0.893204,0.938776,0.915423


{'eval_loss': 0.2724872827529907,
 'eval_accuracy': 0.915,
 'eval_precision': 0.8932038834951457,
 'eval_recall': 0.9387755102040817,
 'eval_f1': 0.9154228855721394}

In [53]:
print(veri_kucuk["label"].value_counts())
print(train_labels[:20])
print(test_labels[:20])

label
0    1024
1     976
Name: count, dtype: int64
[1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
[1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1]


In [54]:
trainer.state.log_history

[{'loss': 0.7616782188415527,
  'grad_norm': 8.172735214233398,
  'learning_rate': 1.936666666666667e-05,
  'epoch': 0.1,
  'step': 20},
 {'loss': 0.6889323234558106,
  'grad_norm': 4.492760181427002,
  'learning_rate': 1.8700000000000004e-05,
  'epoch': 0.2,
  'step': 40},
 {'loss': 0.6903156757354736,
  'grad_norm': 3.7297415733337402,
  'learning_rate': 1.8033333333333332e-05,
  'epoch': 0.3,
  'step': 60},
 {'loss': 0.6684160709381104,
  'grad_norm': 6.493052005767822,
  'learning_rate': 1.7366666666666668e-05,
  'epoch': 0.4,
  'step': 80},
 {'loss': 0.5684875965118408,
  'grad_norm': 5.666116714477539,
  'learning_rate': 1.67e-05,
  'epoch': 0.5,
  'step': 100},
 {'loss': 0.473414945602417,
  'grad_norm': 6.023988246917725,
  'learning_rate': 1.6033333333333335e-05,
  'epoch': 0.6,
  'step': 120},
 {'loss': 0.41500306129455566,
  'grad_norm': 14.102923393249512,
  'learning_rate': 1.5366666666666666e-05,
  'epoch': 0.7,
  'step': 140},
 {'loss': 0.3562354803085327,
  'grad_norm':

In [55]:
from sklearn.metrics import confusion_matrix

pred = trainer.predict(test_dataset)
y_pred_ft = pred.predictions.argmax(axis=1)

print(confusion_matrix(test_labels, y_pred_ft))

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[[93 10]
 [ 7 90]]


In [56]:
print(trainer.state.global_step)

600


In [57]:
print(trainer.state.global_step)
print(trainer.state.epoch)
trainer.state.log_history

600
3.0


[{'loss': 0.7616782188415527,
  'grad_norm': 8.172735214233398,
  'learning_rate': 1.936666666666667e-05,
  'epoch': 0.1,
  'step': 20},
 {'loss': 0.6889323234558106,
  'grad_norm': 4.492760181427002,
  'learning_rate': 1.8700000000000004e-05,
  'epoch': 0.2,
  'step': 40},
 {'loss': 0.6903156757354736,
  'grad_norm': 3.7297415733337402,
  'learning_rate': 1.8033333333333332e-05,
  'epoch': 0.3,
  'step': 60},
 {'loss': 0.6684160709381104,
  'grad_norm': 6.493052005767822,
  'learning_rate': 1.7366666666666668e-05,
  'epoch': 0.4,
  'step': 80},
 {'loss': 0.5684875965118408,
  'grad_norm': 5.666116714477539,
  'learning_rate': 1.67e-05,
  'epoch': 0.5,
  'step': 100},
 {'loss': 0.473414945602417,
  'grad_norm': 6.023988246917725,
  'learning_rate': 1.6033333333333335e-05,
  'epoch': 0.6,
  'step': 120},
 {'loss': 0.41500306129455566,
  'grad_norm': 14.102923393249512,
  'learning_rate': 1.5366666666666666e-05,
  'epoch': 0.7,
  'step': 140},
 {'loss': 0.3562354803085327,
  'grad_norm':

In [58]:
test_result = trainer.evaluate(test_dataset)
print(test_result)

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.071310,0.311894,3,0.915000,0.900000,0.927835,0.913706


{'eval_loss': 0.31189435720443726, 'eval_accuracy': 0.915, 'eval_precision': 0.9, 'eval_recall': 0.9278350515463918, 'eval_f1': 0.9137055837563451}


In [59]:
test_result = trainer.evaluate(test_dataset)
print(test_result)

c:\venv\bert\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.071310,0.311894,3,0.915000,0.900000,0.927835,0.913706


{'eval_loss': 0.31189435720443726, 'eval_accuracy': 0.915, 'eval_precision': 0.9, 'eval_recall': 0.9278350515463918, 'eval_f1': 0.9137055837563451}
